# Solutions · 13 · Digital control design

Worked solutions to the exercises of [notebook 13](../notebooks/13_digital_control_design.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, discrete as dc, design as dg, statespace as sp, response as rs, frequency as fr, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Implement the DC motor position loop as a discrete LQR with integral action at dt = 0.02 (augment the ZOH model with a discrete integrator of the error) and compare its tracking of a 1 rad step and its rejection of a load torque with the emulated lead design.

In [2]:
mot = models.dc_motor(position=True)["system"]
dt = 0.02
Gd = dc.c2d(mot, dt); Ad, Bd, Cd = Gd.A, Gd.B, Gd.C
# augment with a discrete integrator of the tracking error: xi[k+1] = xi[k] + dt (r - y)
Aa = np.block([[Ad, np.zeros((3, 1))], [-dt * Cd, np.eye(1)]]); Ba = np.vstack([Bd, [[0.0]]])
Qa = np.diag([1.0, 10.0, 1000.0, 1e5]); Ra = np.array([[0.1]])
r = sp.lqr(Aa, Ba, Qa, Ra, discrete=True); Ka = r["K"]
Acl = Aa - Ba @ Ka
cl_r = sy.ss(Acl, np.array([[0.0], [0.0], [0.0], [dt]]), np.hstack([Cd, [[0.0]]]), 0.0, dt)
t, y = rs.step(cl_r, 3.0)
cl_d = sy.ss(Acl, np.vstack([Bd, [[0.0]]]), np.hstack([Cd, [[0.0]]]), 0.0, dt)     # a torque-like disturbance through the input
td, yd = rs.step(cl_d, 3.0)
info = rs.step_info(t, y, final=1.0)
print(f"discrete LQR + integral: step overshoot {info['overshoot']:.1f} %, settling {info['settling_time']:.2f} s, final {y[-1]:.4f}; disturbance step rejected to {yd[-1]:.1e} (integral action)")
C = dg.design_lead(mot.to_tf(), 5.0, 50.0)["controller"]; Cz = dc.c2d(C, dt, "tustin")
Pd = dc.c2d(mot.to_tf(), dt); Td = sy.feedback(sy.series(Cz, Pd)); PSd = sy.feedback(Pd, Cz)
t2, y2 = rs.step(Td, 3.0); td2, yd2 = rs.step(PSd, 3.0)
print(f"emulated lead: step overshoot {rs.step_info(t2, y2, final=1.0)['overshoot']:.1f} %, settling {rs.step_info(t2, y2, final=1.0)['settling_time']:.2f} s; disturbance leaves a steady offset {yd2[-1]:.3f} (no integral action)")

discrete LQR + integral: step overshoot 4.7 %, settling 1.42 s, final 1.0001; disturbance step rejected to -7.8e-07 (integral action)
emulated lead: step overshoot 13.8 %, settling 1.34 s; disturbance leaves a steady offset 0.024 (no integral action)


The discrete LQR with an integrator state removes the offset a load disturbance would leave (the lead
compensator leaves one, having no integrator beyond the plant's own, which is before the disturbance
enters). Direct discrete design on the ZOH model with the augmented state is the digital counterpart of
notebook 10's integral LQR, and needs no continuous design at all.

## Exercise 2

Add a one-sample computation delay to the plant model (a state z^-1) and redesign the discrete pole placement for the delayed model. How does the achievable speed compare with the delay-free design at dt = 0.02 and at dt = 0.1?

In [3]:
mot = models.dc_motor(position=True)["system"]
for dt in (0.02, 0.1):
    Gd = dc.c2d(mot, dt); Ad, Bd = Gd.A, Gd.B
    # one sample of computation delay: u applied at k+1 -> add a state holding the last u
    Adel = np.block([[Ad, Bd], [np.zeros((1, 3)), np.zeros((1, 1))]]); Bdel = np.vstack([np.zeros((3, 1)), [[1.0]]])
    for s_poles in ([-8 + 6j, -8 - 6j, -20.0], [-16 + 12j, -16 - 12j, -40.0]):
        z_poles = np.exp(np.array(s_poles) * dt)
        K0 = sp.place(Ad, Bd, z_poles)
        Kd = sp.place(Adel, Bdel, np.r_[z_poles, 0.0])                   # the delay state's pole at 0 (dead-beat on the delay)
        ok0 = np.max(np.abs(np.linalg.eigvals(Ad - Bd @ K0))) < 1
        print(f"dt = {dt}, target poles {np.round(s_poles, 0)}: delay-free gain |K| = {np.linalg.norm(K0):.1f}; with the delay state |K| = {np.linalg.norm(Kd):.1f}, max |z| = {np.max(np.abs(np.linalg.eigvals(Adel - Bdel @ Kd))):.4f}")

dt = 0.02, target poles [ -8.+6.j  -8.-6.j -20.+0.j]: delay-free gain |K| = 794.0; with the delay state |K| = 794.2, max |z| = 0.8521
dt = 0.02, target poles [-16.+12.j -16.-12.j -40. +0.j]: delay-free gain |K| = 4528.7; with the delay state |K| = 4530.6, max |z| = 0.7261
dt = 0.1, target poles [ -8.+6.j  -8.-6.j -20.+0.j]: delay-free gain |K| = 348.7; with the delay state |K| = 349.0, max |z| = 0.4493
dt = 0.1, target poles [-16.+12.j -16.-12.j -40. +0.j]: delay-free gain |K| = 769.3; with the delay state |K| = 769.9, max |z| = 0.2019


Modelling the computation delay as an extra state lets the pole placement account for it exactly, at the
price of larger gains, which grow with the target speed and with the sampling period: at Δt = 0.1 and the
faster targets the delayed design needs gains several times the delay-free ones. A delay of one sample is a
larger fraction of the loop's time scale when the samples are long - the same lesson as the half-sample delay
of the hold.

## Exercise 3

**Implement it yourself:** write the discrete PID as a difference equation (u[k] from e[k], e[k-1], e[k-2] and u[k-1], u[k-2]) from the coefficients of `discrete.pid_discrete` and verify it against `response.step` of the controller transfer function.

In [4]:
dt = 0.02
Cz = dc.pid_discrete(3.0, 2.0, 0.3, dt)
b, a = Cz.num, Cz.den                                      # u[k] = (b0 e[k] + b1 e[k-1] + b2 e[k-2] - a1 u[k-1] - a2 u[k-2]) / a0
e = np.ones(200); u = np.zeros(200)
for k in range(200):
    acc = sum(b[i] * e[k - i] for i in range(len(b)) if k - i >= 0) - sum(a[i] * u[k - i] for i in range(1, len(a)) if k - i >= 0)
    u[k] = acc / a[0]
t, u_lib = rs.step(Cz, (200 - 1) * dt)
print(f"difference equation vs the library's step response of C(z): {np.max(np.abs(u - u_lib)):.1e}")
print("coefficients: b =", np.round(b, 4), " a =", np.round(a, 4))

difference equation vs the library's step response of C(z): 4.2e-13
coefficients: b = [ 13.04   -24.0133  11.    ]  a = [ 1.     -1.3333  0.3333]


The transfer function's coefficients are the difference equation's: the numerator weights the error's history,
the denominator the control's own. Four multiplications and a few additions per sample - which is what runs
on the microcontroller, and why the discretised controller, not the continuous design, is the one to test.